In [ ]:
import numpy as np
import tensorflow as tf
print(tf.__version__) 
print(tf.keras.__version__) 

In [ ]:
from tensorflow.keras import mixed_precision
import tensorflow as tf

# Check if mixed precision can be set
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

print("Mixed precision set to:", mixed_precision.global_policy())

In [ ]:
from tensorflow.keras import mixed_precision
import os
from tflite_model_maker import object_detector
from tflite_model_maker import model_spec
from sklearn.model_selection import train_test_split
import shutil
import glob

In [ ]:
split_dirs = {
    'train': {'images': 'split/10-25m/train/images', 'annotations': 'split/10-25m/train/annotations'},
    'val': {'images': 'split/10-25m/val/images', 'annotations': 'split/10-25m/val/annotations'},
}

# Load the data for each split using Pascal VOC format
train_data = object_detector.DataLoader.from_pascal_voc(
    split_dirs['train']['images'], 
    split_dirs['train']['annotations'], 
    label_map={1: "mobil"}
)
val_data = object_detector.DataLoader.from_pascal_voc(
    split_dirs['val']['images'], 
    split_dirs['val']['annotations'], 
    label_map={1: "mobil"}
)

# Load EfficientDet Lite 0 model specification
spec = model_spec.get('efficientdet_lite0')

# Optional: Freeze part of the model (efficientnet backbone)
spec.config.var_freeze_expr = 'efficientnet'
spec.config.max_input_size = 224  # Reduce input image size to 224x224

Train

In [ ]:
model = object_detector.create(
    train_data, 
    model_spec=spec, 
    validation_data=val_data, 
    epochs=120, 
    batch_size=4, 
    train_whole_model=True
)

# Export the trained model to the current directory
model.export(export_dir=os.path.join("MyModel","50m"))

Test

In [ ]:
test_dirs = {
    'test': {'images': 'split/300m/test/images', 'annotations': 'split/300m/test/annotations'}
}

test_data = object_detector.DataLoader.from_pascal_voc(
    test_dirs['test']['images'], 
    test_dirs['test']['annotations'], 
    label_map={1: "mobil"}
)

In [ ]:
# Evaluate the model on the test data
model.evaluate(test_data, batch_size=8)